# Gemma как отдельный сервис в DataSphere

Код сервиса находится в `llm_gemma_service/` этого репозитория. Веса, сборка,
лог и сведения о процессе сохраняются отдельно в `/home/jupyter/project/llm/cuda122`.

1. Если бинарник и веса уже готовы, переходите к шагу 3.
2. При первом запуске выполните настройку и скачивание (шаги 1–2).
3. Подключитесь к серверу: проверка `/health`, готовый сервер используется повторно;
   при отсутствии запускается новый процесс. Импорт модуля сам по себе ничего не запускает.
4. В `inference.ipynb` поставьте `USE_LLM_SERVICE = True`.

Ноутбуки должны выполняться на той же GPU-ВМ: общий диск проекта не означает общий `localhost`.
Обновление Python-кода и повторный запуск ячеек pipeline не перезапускают LLM.
При остановке/замене ВМ DataSphere модель нужно загрузить в GPU заново.

Это ноутбук настройки для Linux x86_64, NVIDIA L4, драйвера 535 и CUDA 12.2.
Он не гарантирует работу на другой GPU и не удерживает ВМ включённой 24/7.
`GEMMA_ROOT` и `GEMMA_PORT` можно задать в окружении **до импорта** модуля;
для существующего сервера оставьте прежние значения.


## 1. Установить инструменты и собрать сервер

Без sudo; первая сборка показывает прогресс и сохраняет полный лог `build.log`. Не перезапускайте ячейку, пока она выполняется.

In [ ]:
import importlib.util
import json
import os
import platform
import re
import shutil
import signal
import socket
import subprocess
import sys
import tarfile
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

assert platform.system() == "Linux" and platform.machine() == "x86_64", "Запускайте в Linux-ноутбуке DataSphere."
PROJECT = Path("/home/jupyter/project")
assert PROJECT.is_dir(), "Не найдено хранилище /home/jupyter/project."
ROOT = Path(os.environ.get("GEMMA_ROOT", str(PROJECT / "llm" / "cuda122")))
ROOT.mkdir(parents=True, exist_ok=True)
MAMBA = ROOT / "bin" / "micromamba"
TOOLCHAIN = ROOT / "toolchain"
SOURCE = ROOT / "llama.cpp"
BUILD = ROOT / "build"
SERVER = BUILD / "bin" / "llama-server"
TAG = "b11429"
BUILD_LOG = ROOT / "build.log"
SERVER_LOG = ROOT / "server.log"
PORT = int(os.environ.get("GEMMA_PORT", "11435"))
BASE = f"http://127.0.0.1:{PORT}"

print(subprocess.check_output(["nvidia-smi"], text=True))
gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,compute_cap", "--format=csv,noheader"], text=True
)
print(gpu_info)
first_gpu = gpu_info.splitlines()[0].split(",")
assert "L4" in first_gpu[0] and first_gpu[2].strip() == "8.9", "Эта сборка предназначена для вашей NVIDIA L4 (SM 8.9)."
assert int(first_gpu[1].strip().split(".")[0]) >= 525, "Для CUDA 12.2 требуется подходящий драйвер семейства 525 или новее."

ENV = os.environ.copy()
ENV["MAMBA_ROOT_PREFIX"] = str(ROOT / "mamba-cache")
# Never add CUDA's lib/stubs here: the real GPU driver belongs to the VM.
ENV["LD_LIBRARY_PATH"] = ":".join([
    str(BUILD / "bin"), str(TOOLCHAIN / "lib"),
    str(TOOLCHAIN / "targets" / "x86_64-linux" / "lib"),
    os.environ.get("LD_LIBRARY_PATH", ""),
])

def run_logged(command):
    """Stream build output; preserve the complete log on the project disk."""
    with BUILD_LOG.open("a") as log:
        with subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            if process.wait() != 0:
                raise RuntimeError(f"Ошибка подготовки. Полный лог: {BUILD_LOG}")

def tool_command(*arguments):
    return [str(MAMBA), "run", "-p", str(TOOLCHAIN), *map(str, arguments)]

if not (ROOT / ".build-ready").is_file() or not SERVER.is_file():
    if shutil.disk_usage(ROOT).free < 45 * 1024**3:
        raise RuntimeError("Для первой установки и весов нужно не менее 45 GiB свободного места.")
    if not MAMBA.is_file():
        archive = ROOT / "micromamba.tar.bz2"
        subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(archive),
                        "https://micro.mamba.pm/api/micromamba/linux-64/latest"], check=True)
        # Extract only the executable, without trusting archive paths.
        with tarfile.open(archive, "r:bz2") as package:
            member = package.extractfile("bin/micromamba")
            assert member is not None, "В архиве нет micromamba."
            MAMBA.parent.mkdir(exist_ok=True)
            with MAMBA.open("wb") as destination:
                shutil.copyfileobj(member, destination)
        MAMBA.chmod(0o755)
        archive.unlink()
    if not (TOOLCHAIN / ".ready").is_file():
        print("Установка CUDA 12.2 и компилятора в хранилище проекта (без sudo)...", flush=True)
        action = "install" if (TOOLCHAIN / "conda-meta").is_dir() else "create"
        run_logged([
            str(MAMBA), action, "-y", "-p", str(TOOLCHAIN), "--override-channels",
            "-c", "nvidia/label/cuda-12.2.2", "-c", "conda-forge", "--strict-channel-priority",
            "cuda-toolkit=12.2.2", "cmake=3.31", "ninja", "git", "gxx_linux-64=11",
        ])
        (TOOLCHAIN / ".ready").write_text("CUDA 12.2.2; GCC 11\n")
        # Remove downloaded package archives, keeping the installed environment.
        run_logged([str(MAMBA), "clean", "--tarballs", "-y"])
    run_logged(tool_command("nvcc", "--version"))
    if not (SOURCE / ".git").is_dir():
        run_logged(tool_command("git", "clone", "--depth", "1", "--branch", TAG,
                                "https://github.com/ggml-org/llama.cpp.git", SOURCE))
    commit = subprocess.check_output(tool_command("git", "-C", SOURCE, "rev-parse", "HEAD"), env=ENV, text=True).strip()
    compiler_c = TOOLCHAIN / "bin" / "x86_64-conda-linux-gnu-cc"
    compiler_cpp = TOOLCHAIN / "bin" / "x86_64-conda-linux-gnu-c++"
    assert compiler_c.exists() and compiler_cpp.exists(), "Не найден компилятор GCC из окружения."
    print("Сборка llama.cpp. Первый запуск может быть длительным; прогресс ниже.", flush=True)
    run_logged(tool_command(
        "cmake", "-S", SOURCE, "-B", BUILD, "-G", "Ninja",
        "-DCMAKE_BUILD_TYPE=Release", "-DGGML_CUDA=ON", "-DGGML_NATIVE=OFF",
        f"-DCUDAToolkit_ROOT={TOOLCHAIN}",
        f"-DCMAKE_CUDA_COMPILER={TOOLCHAIN / 'bin' / 'nvcc'}",
        f"-DCMAKE_C_COMPILER={compiler_c}", f"-DCMAKE_CXX_COMPILER={compiler_cpp}",
        f"-DCMAKE_CUDA_HOST_COMPILER={compiler_cpp}",
        "-DCMAKE_CUDA_ARCHITECTURES=89-real", "-DLLAMA_OPENSSL=OFF",
        "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF", "-DLLAMA_BUILD_APP=OFF",
        "-DLLAMA_BUILD_UI=OFF", "-DLLAMA_BUILD_SERVER=ON",
    ))
    run_logged(tool_command("cmake", "--build", BUILD, "--target", "llama-server", "--parallel", "2"))
    run_logged(tool_command(SERVER, "--version"))
    (ROOT / "build-info.json").write_text(json.dumps({"tag": TAG, "commit": commit, "cuda": "12.2.2"}, indent=2))
    (ROOT / ".build-ready").write_text(commit)

devices = subprocess.check_output(tool_command(SERVER, "--list-devices"), env=ENV, stderr=subprocess.STDOUT, text=True)
print(devices)
assert "CUDA0" in devices, "CUDA-устройство не найдено. Не продолжаем тест на CPU."
print("Сервер собран, CUDA доступна. Драйвер ВМ не изменялся.")


## 2. Скачать веса один раз

Скачивается только текстовая модель Q4_0 (~14.4 GB), без мультимодального проектора. Ревизия фиксируется в `weights/revision.txt`. Если Hugging Face вернёт 401/403, проверьте доступ к модели под своим аккаунтом; токен не отправляйте в чат.

In [ ]:
if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub"], check=True)
from huggingface_hub import HfApi, hf_hub_download

REPO = "google/gemma-4-26B-A4B-it-qat-q4_0-gguf"
FILENAME = "gemma-4-26B_q4_0-it.gguf"
WEIGHTS = ROOT / "weights"
WEIGHTS.mkdir(exist_ok=True)
MODEL_PATH = WEIGHTS / FILENAME
REVISION_FILE = WEIGHTS / "revision.txt"
if not REVISION_FILE.exists():
    revision = HfApi().model_info(REPO).sha
    REVISION_FILE.write_text(revision)
revision = REVISION_FILE.read_text().strip()
started = time.perf_counter()
MODEL_PATH = Path(hf_hub_download(
    repo_id=REPO, filename=FILENAME, revision=revision,
    local_dir=WEIGHTS,
))
assert MODEL_PATH.is_file() and MODEL_PATH.stat().st_size > 10 * 1024**3, "Весовой файл неполный."
print(f"Веса: {MODEL_PATH}\nРазмер: {MODEL_PATH.stat().st_size / 1024**3:.2f} GiB")
print(f"Скачивание / проверка кэша: {time.perf_counter() - started:.1f} с. Это время не входит в замер загрузки GPU.")


## 3. Подключиться или запустить

Эту ячейку можно повторять: исправный сервер сохраняется. Если загрузка уже началась другим вызовом, новая копия модели не создаётся. При прерывании ожидания сервер продолжает запускаться; выполните ячейку ещё раз, чтобы подключиться.

Если сервер был запущен прежним ноутбуком, готовый API также будет использован. Команда остановки нового файла управляет только процессами, которые он сам запустил.

In [ ]:
import sys
from pathlib import Path

# Notebook can be opened from the repository root or llm_gemma_service/.
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "llm_gemma_service" / "gemma_service.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Откройте ноутбук из папки этого репозитория.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from llm_gemma_service import gemma_service as llm

ensure_running = llm.ensure_running
measure_request = llm.measure_request
stop_model = llm.stop_model
connection = ensure_running()
print(connection)


## 4. Отправлять запросы и менять ранкер

Повторяйте эту ячейку или вызывайте `measure_request()` из своего кода. Она сама проверяет готовность сервера. Менять промпты и код ранкера можно без выгрузки весов. Не вызывайте старую `benchmark_load()` — она предназначена для принудительного перезапуска и замера загрузки.

In [ ]:
result = measure_request(
    "Коротко посоветуй музыку для спокойной работы.",
    max_tokens=96,
)
# result["seconds"] — время до полного ответа, без запуска/загрузки модели.
# result["text"] — текст ответа.


## 5. Использование из pipeline и остановка

В `inference.ipynb` включите `USE_LLM_SERVICE = True`. Роли сообщений и лимит длины
ответа передаются через `measure_chat`. `measure_request` остаётся для ручных запросов:

```python
from llm_gemma_service import measure_request
result = measure_request("Посоветуй музыку для работы.", max_tokens=128)
```

При запуске из подпапки добавьте корень репозитория в `sys.path`, как в шаге 3.
Из терминала в корне репозитория:

```bash
python llm_gemma_service/gemma_service.py ensure
python llm_gemma_service/gemma_service.py status
```

Остановка выполняется явно: `stop_model()` или команда с действием `stop`.
Не добавляйте остановку в обычные ячейки pipeline. Сервер продолжает работать после
завершения ячейки, пока жива ВМ; повторное подключение не загружает веса заново.


In [ ]:
print(llm.status())
# stop_model()  # Раскомментировать только если нужно остановить LLM и освободить GPU.
